In [ ]:
!pip install -q peft
!pip install pytorch-metric-learning
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# SOURCE_DIR = "/content/drive/MyDrive/캡스톤 디자인 - Team Oasis"
# ZIP_PATH = "images_final.zip"

# print("📦 구글 드라이브 내에서 압축을 시작합니다...")

# !cd "{SOURCE_DIR}" && zip -r -q "{ZIP_PATH}" "images_final"

# print(f"✅ 압축 완료! 만들어진 파일: {ZIP_PATH}")

# ZIP_PATH = "/content/drive/MyDrive/캡스톤 디자인 - Team Oasis/images_final.zip"

# # 1. 드라이브의 zip 파일을 코랩 로컬(/content)로 복사
# !cp "{ZIP_PATH}" /content/

# # 2. 로컬 디스크에서 압축을 해제
# # -q 옵션: 수천 줄의 텍스트 출력을 숨겨서 브라우저 렉을 방지
!unzip -q "/content/drive/MyDrive/캡스톤 디자인 - Team Oasis/images_final" -d /content/images/

In [ ]:
# from google.colab import files
# file_path = '/content/images_final.zip'

# files.download(file_path)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import DistilBertModel, AutoTokenizer, CLIPVisionModelWithProjection, CLIPProcessor
from peft import LoraConfig, get_peft_model
from pytorch_metric_learning import losses # 💡 SupCon 추가

import torchvision.transforms as T

# 학습용(Train) 데이터 증강 파이프라인
train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.8, 1.0)), # 사진의 80~100% 크기로 무작위로 확대해서 자르기
    T.RandomHorizontalFlip(p=0.5),              # 50% 확률로 사진을 좌우 반전 (풍경 사진에 최고!)
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02) # 빛, 대비, 채도 아주 살짝 비틀기
])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
STAGE1_PATH = '/content/drive/MyDrive/checkpoints/stage1/best.pt'
LR = 1e-5

TEXT_MODEL_NAME = 'sentence-transformers/clip-ViT-B-32-multilingual-v1'
VISION_MODEL_NAME = 'openai/clip-vit-base-patch32'

text_encoder = DistilBertModel.from_pretrained(TEXT_MODEL_NAME)
vision_encoder = CLIPVisionModelWithProjection.from_pretrained(VISION_MODEL_NAME)

text_lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=["q_lin", "v_lin"], lora_dropout=0.1)
text_encoder = get_peft_model(text_encoder, text_lora_config)

vision_lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], lora_dropout=0.1)
vision_encoder = get_peft_model(vision_encoder, vision_lora_config)

class ProjectionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 256))
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

class SceneHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256, 13)
        )
    def forward(self, x):
        return self.net(x)

txt_proj_layer = nn.Linear(768, 512).to(device)
proj_head = ProjectionHead().to(device)
scene_head = SceneHead().to(device)

checkpoint = torch.load(STAGE1_PATH, map_location=device)
txt_proj_layer.load_state_dict(checkpoint['txt_proj_layer'])
proj_head.load_state_dict(checkpoint['proj_head'])
scene_head.load_state_dict(checkpoint['cls_head'])

text_encoder.to(device); vision_encoder.to(device)

params = list(text_encoder.parameters()) + list(vision_encoder.parameters()) + \
         list(txt_proj_layer.parameters()) + list(proj_head.parameters()) + \
         list(scene_head.parameters())
optimizer = torch.optim.AdamW(params, lr=LR)

# 💡 SupCon Loss 함수 선언
supcon_loss_fn = losses.SupConLoss(temperature=0.05).to(device)

def train_step(images, input_ids, masks, labels):
    optimizer.zero_grad()

    img_feat = vision_encoder(pixel_values=images).image_embeds
    txt_out = text_encoder(input_ids=input_ids, attention_mask=masks)[0][:, 0, :]
    txt_feat = txt_proj_layer(txt_out)

    i_emb = proj_head(img_feat)
    t_emb = proj_head(txt_feat)

    # 🚨 1) L_CLIP: 이미지 뇌와 텍스트 뇌 다리 연결! (수식 복구)
    logits = i_emb @ t_emb.T * (1.0 / 0.07)
    targets = torch.arange(images.size(0)).to(device)
    loss_clip = (F.cross_entropy(logits, targets) + F.cross_entropy(logits.T, targets)) / 2

    # 2) L_CE: 장소 분류
    scene_logits = scene_head(img_feat)
    loss_ce = F.cross_entropy(scene_logits, labels)

    # 🚨 3) L_SupCon: 같은 장소끼리 예쁘게 뭉치기!
    loss_supcon = supcon_loss_fn(i_emb, labels)

    # 최종 통합 Loss (비율 조절: 분류에 너무 집착하지 않게 낮추고 supcon 비율 증)
    total_loss = (1.0 * loss_clip) + (2.0 * loss_supcon) + (0.1 * loss_ce)

    total_loss.backward()
    optimizer.step()
    return total_loss.item()

print("🚀 Stage 2: LoRA Fine-tuning (SupCon 장착) 준비 완료!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/539M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.final_layer_norm.weight                           | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_no

🚀 Stage 2: LoRA Fine-tuning (SupCon 장착) 준비 완료!


In [ ]:
# Cell 3. 설정값
# ====================================================
# 경로
CSV_PATH   = '/content/drive/MyDrive/metadata_stage1_ready.xlsx'
IMAGE_ROOT = '/content/images/images_final'
SAVE_PATH  = '/content/drive/MyDrive/checkpoints/stage1/'
# ====================================================

# 차원
IMG_DIM  = 512   # CLIP 이미지 인코더 출력
TXT_DIM  = 768   # DistilBERT 출력
PROJ_DIM = 256   # ProjectionHead 출력

# scene 라벨 (13개)
SCENE_CLASSES = ['바다','산','숲','도시','호수','거리','전통','공원','전망','조형물','카페','온천','전시']
SCENE2IDX     = {s: i for i, s in enumerate(SCENE_CLASSES)}
NUM_CLASSES   = len(SCENE_CLASSES)

# 하이퍼파라미터
BATCH_SIZE = 128
NUM_EPOCHS = 20
LR         = 1e-3
LAMBDA_CE  = 0.7

print(f'NUM_CLASSES: {NUM_CLASSES}')
print(f'IMG_DIM: {IMG_DIM}, TXT_DIM: {TXT_DIM}, PROJ_DIM: {PROJ_DIM}')

NUM_CLASSES: 13
IMG_DIM: 512, TXT_DIM: 768, PROJ_DIM: 256


In [ ]:
processor      = CLIPProcessor.from_pretrained(VISION_MODEL_NAME)

TEXT_MODEL_NAME   = 'sentence-transformers/clip-ViT-B-32-multilingual-v1'

text_encoder   = DistilBertModel.from_pretrained(TEXT_MODEL_NAME).to(device)
text_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/371 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
import os
import unicodedata
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import pandas as pd

# --- [추가] 한글 인코딩 문제 해결을 위한 정규화 함수 ---
def dn(text):
    if pd.isna(text): return ""
    return unicodedata.normalize('NFC', str(text)).strip()

# --- [추가] 실제 폴더 안의 파일들을 미리 스캔 (딱 한 번만 실행) ---
# 엑셀의 이름과 실제 파일명이 인코딩 차이로 다를 때를 대비해 매핑 테이블을 만듭니다.
print("🔎 이미지 폴더 스캔 중...")
actual_file_map = {dn(f): f for f in os.listdir(IMAGE_ROOT) if not f.startswith('.')}
print(f"✅ 총 {len(actual_file_map)}개의 실제 파일 인식 완료.")

class TravelDataset(Dataset):
    def __init__(self, df, image_root, processor, tokenizer, scene2idx,transform):
        self.df         = df.reset_index(drop=True)
        self.image_root = image_root
        self.processor  = processor
        self.tokenizer  = tokenizer
        self.scene2idx  = scene2idx
        self.transform  = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 1. 엑셀에 적힌 이미지 아이디 정규화
        target_id = dn(row['image_id'])

        # 2. 매핑 테이블에서 실제 파일명 찾기
        real_file_name = actual_file_map.get(target_id)

        if real_file_name is None:
            # 파일이 없을 경우 에러 메시지를 명확히 출력
            raise FileNotFoundError(f"❌ 파일 없음: {target_id} (엑셀 행 번호: {idx})")

        img_path = os.path.join(self.image_root, real_file_name)

        # 이미지 로드 및 처리
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform is not None:
                image = self.transform(image)
            pixel_values = self.processor(images=image, return_tensors='pt')['pixel_values'].squeeze(0)
        except Exception as e:
            print(f"❌ 이미지 로드 실패: {img_path}")
            raise e

        # 텍스트 처리
        enc = self.tokenizer(
            str(row['caption']),
            padding='max_length', max_length=64,
            truncation=True, return_tensors='pt'
        )
        input_ids      = enc['input_ids'].squeeze(0)
        attention_mask = enc['attention_mask'].squeeze(0)

        # scene 라벨
        scene_label = self.scene2idx.get(str(row['scene']), 0)

        return pixel_values, input_ids, attention_mask, scene_label

# --- 데이터 로드 및 분할 ---
df = pd.read_excel(CSV_PATH)

# [중요] 엑셀의 image_id가 실제 파일과 매칭되는지 전수 검사 (미리 에러 방지)
df['is_file_exist'] = df['image_id'].apply(lambda x: dn(x) in actual_file_map)
missing_files = df[df['is_file_exist'] == False]

if len(missing_files) > 0:
    print(f"⚠️ 경고: 엑셀 명단 중 {len(missing_files)}개의 파일이 폴더에 없습니다!")
    print(f"예시: {missing_files['image_id'].iloc[0]}")
    # 파일이 없는 행은 학습에서 아예 제외
    df = df[df['is_file_exist'] == True].copy()

print('\nscene 분포:')
print(df['scene'].value_counts())

unknown = set(df['scene'].unique()) - set(SCENE_CLASSES)
if unknown:
    print(f'\n⚠️ SCENE_CLASSES에 없는 값: {unknown}')

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
print(f'\n최종 Train: {len(train_df)}장 / Val: {len(val_df)}장')

# 데이터로더 생성
train_dataset = TravelDataset(train_df, IMAGE_ROOT, processor, text_tokenizer, SCENE2IDX,transform=train_transform)
val_dataset   = TravelDataset(val_df,   IMAGE_ROOT, processor, text_tokenizer, SCENE2IDX,transform=None)

# num_workers=2가 가끔 코랩에서 에러를 내면 0으로 바꾸세요.
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train 배치 수: {len(train_loader)} / Val 배치 수: {len(val_loader)}')

🔎 이미지 폴더 스캔 중...
✅ 총 4831개의 실제 파일 인식 완료.
⚠️ 경고: 엑셀 명단 중 71개의 파일이 폴더에 없습니다!
예시: 숨도_5_공공3유형.jpg

scene 분포:
scene
전통     880
공원     819
바다     555
거리     331
숲      282
도시     270
산      242
전망     226
호수     214
조형물    212
카페     137
전시      18
온천       3
Name: count, dtype: int64

최종 Train: 3770장 / Val: 419장
Train 배치 수: 30 / Val 배치 수: 4


In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import silhouette_score

print("📊 [최종 결전] 3단계 모델 종합 성적표 (Loss, Accuracy, Silhouette) 추출 중...\n")

# ==========================================
# 1. 경로 설정 및 파일 자동 탐색
# ==========================================
STAGE1_DIR = "/content/drive/MyDrive/checkpoints/stage1"
STAGE2_DIR = "/content/drive/MyDrive/checkpoints/stage2"

def find_pt_file(directory):
    if not os.path.exists(directory): return None
    pt_files = [f for f in os.listdir(directory) if f.endswith('.pt')]
    return pt_files[0] if pt_files else None

file1 = find_pt_file(STAGE1_DIR)
file2 = find_pt_file(STAGE2_DIR)

# ==========================================
# 2. Stage 1 & Stage 2의 모든 '뇌(Heads)' 불러오기
# ==========================================
def load_heads(file_path):
    p_head = ProjectionHead().to(device)
    t_proj = nn.Linear(768, 512).to(device)
    s_head = SceneHead().to(device)

    checkpoint = torch.load(file_path, map_location=device)

    # 가중치 덮어쓰기
    p_head.load_state_dict(checkpoint['proj_head'])

    if 'txt_proj_layer' in checkpoint:
        t_proj.load_state_dict(checkpoint['txt_proj_layer'])

    if 'cls_head' in checkpoint:
        s_head.load_state_dict(checkpoint['cls_head'])
    elif 'scene_head' in checkpoint:
        s_head.load_state_dict(checkpoint['scene_head'])

    p_head.eval(); t_proj.eval(); s_head.eval()
    return p_head, t_proj, s_head

print("🧠 Stage 1, Stage 2 가중치 로드 중...")
p_head_s1, t_proj_s1, s_head_s1 = load_heads(os.path.join(STAGE1_DIR, file1))
p_head_s2, t_proj_s2, s_head_s2 = load_heads(os.path.join(STAGE2_DIR, file2))

vision_encoder.eval()
text_encoder.eval()

# ==========================================
# 3. 평가 메인 루프 (400장 동시 채점)
# ==========================================
print("🚀 400장의 시험지로 종합 평가를 시작합니다...")

metrics = {
    'base': {'embs': [], 'labels': []},
    's1':   {'loss': 0, 'correct': 0, 'embs': [], 'labels': []},
    's2':   {'loss': 0, 'correct': 0, 'embs': [], 'labels': []}
}
total_samples = 0
TEMPERATURE = 0.05
CE_WEIGHT = 0.2

with torch.no_grad():
    for batch in val_loader:
        pixel_values, input_ids, attention_mask, scene_labels = [b.to(device) for b in batch]
        batch_size = pixel_values.size(0)
        total_samples += batch_size
        targets = torch.arange(batch_size).to(device)

        # 공통 특징 추출 (Baseline)
        img_feat = vision_encoder(pixel_values=pixel_values).image_embeds
        txt_out = text_encoder(input_ids=input_ids, attention_mask=attention_mask)[0][:, 0, :]

        metrics['base']['embs'].append(img_feat.cpu().numpy())
        metrics['base']['labels'].append(scene_labels.cpu().numpy())

        # ----------------------------------
        # 🟢 Stage 1 채점
        # ----------------------------------
        i_emb_s1 = F.normalize(p_head_s1(img_feat), p=2, dim=-1)
        t_emb_s1 = F.normalize(p_head_s1(t_proj_s1(txt_out)), p=2, dim=-1)
        scene_logits_s1 = s_head_s1(img_feat)

        # Loss
        logits_s1 = i_emb_s1 @ t_emb_s1.T * (1.0 / TEMPERATURE)
        loss_clip_s1 = (F.cross_entropy(logits_s1, targets) + F.cross_entropy(logits_s1.T, targets)) / 2
        loss_ce_s1 = F.cross_entropy(scene_logits_s1, scene_labels)
        metrics['s1']['loss'] += (loss_clip_s1 + CE_WEIGHT * loss_ce_s1).item()

        # Acc
        metrics['s1']['correct'] += (torch.argmax(scene_logits_s1, dim=1) == scene_labels).sum().item()
        metrics['s1']['embs'].append(i_emb_s1.cpu().numpy())
        metrics['s1']['labels'].append(scene_labels.cpu().numpy())

        # ----------------------------------
        # 🔴 Stage 2 채점
        # ----------------------------------
        i_emb_s2 = F.normalize(p_head_s2(img_feat), p=2, dim=-1)
        t_emb_s2 = F.normalize(p_head_s2(t_proj_s2(txt_out)), p=2, dim=-1)
        scene_logits_s2 = s_head_s2(img_feat)

        # Loss
        logits_s2 = i_emb_s2 @ t_emb_s2.T * (1.0 / TEMPERATURE)
        loss_clip_s2 = (F.cross_entropy(logits_s2, targets) + F.cross_entropy(logits_s2.T, targets)) / 2
        loss_ce_s2 = F.cross_entropy(scene_logits_s2, scene_labels)
        metrics['s2']['loss'] += (loss_clip_s2 + CE_WEIGHT * loss_ce_s2).item()

        # Acc
        metrics['s2']['correct'] += (torch.argmax(scene_logits_s2, dim=1) == scene_labels).sum().item()
        metrics['s2']['embs'].append(i_emb_s2.cpu().numpy())
        metrics['s2']['labels'].append(scene_labels.cpu().numpy())

# ==========================================
# 4. 최종 계산 및 성적표 출력
# ==========================================
num_batches = len(val_loader)

# Baseline
base_embs = np.concatenate(metrics['base']['embs'], axis=0)
base_labels = np.concatenate(metrics['base']['labels'], axis=0)
sil_base = silhouette_score(base_embs, base_labels, metric='cosine')

# Stage 1
s1_loss = metrics['s1']['loss'] / num_batches
s1_acc = (metrics['s1']['correct'] / total_samples) * 100
s1_embs = np.concatenate(metrics['s1']['embs'], axis=0)
sil_s1 = silhouette_score(s1_embs, base_labels, metric='cosine')

# Stage 2
s2_loss = metrics['s2']['loss'] / num_batches
s2_acc = (metrics['s2']['correct'] / total_samples) * 100
s2_embs = np.concatenate(metrics['s2']['embs'], axis=0)
sil_s2 = silhouette_score(s2_embs, base_labels, metric='cosine')

print("\n" + "🏆"*20)
print("     [최종 Ablation Study (절제 연구) 성적표]     ")
print("🏆"*20 + "\n")

print(f"🔹 [1] 튜닝 전 (Baseline CLIP)")
print(f"   - Validation Loss : N/A (분류기 없음)")
print(f"   - Classification  : N/A (제로샷 전용)")
print(f"   - Silhouette Score: {sil_base:.4f} (군집화 X)\n")

print(f"🔹 [2] Stage 1 (텍스트-이미지 1차 정렬)")
print(f"   - Validation Loss : {s1_loss:.4f}")
print(f"   - Classification  : {s1_acc:.2f}%")
print(f"   - Silhouette Score: {sil_s1:.4f}\n")

print(f"🔹 [3] Stage 2 (SupCon 대조 학습 적용 완료) 🚀")
print(f"   - Validation Loss : {s2_loss:.4f} (낮을수록 좋음)")
print(f"   - Classification  : {s2_acc:.2f}% (높을수록 좋음)")
print(f"   - Silhouette Score: {sil_s2:.4f} (높을수록 감성 검색 완벽!)\n")
print("="*45)

📊 [최종 결전] 3단계 모델 종합 성적표 (Loss, Accuracy, Silhouette) 추출 중...

🧠 Stage 1, Stage 2 가중치 로드 중...
🚀 400장의 시험지로 종합 평가를 시작합니다...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆
     [최종 Ablation Study (절제 연구) 성적표]     
🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆

🔹 [1] 튜닝 전 (Baseline CLIP)
   - Validation Loss : N/A (분류기 없음)
   - Classification  : N/A (제로샷 전용)
   - Silhouette Score: 0.0313 (군집화 X)

🔹 [2] Stage 1 (텍스트-이미지 1차 정렬)
   - Validation Loss : 3.5890
   - Classification  : 72.55%
   - Silhouette Score: 0.0262

🔹 [3] Stage 2 (SupCon 대조 학습 적용 완료) 🚀
   - Validation Loss : 2.7339 (낮을수록 좋음)
   - Classification  : 73.75% (높을수록 좋음)
   - Silhouette Score: 0.0594 (높을수록 감성 검색 완벽!)



In [ ]:
import os
import time
from google.colab import drive

EPOCHS = 5
SAVE_DIR = '/content/drive/MyDrive/checkpoints/stage2_1'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"🏃‍♂️ Stage 2 학습을 시작합니다! (총 {EPOCHS} Epochs)")

for epoch in range(EPOCHS):
    start_time = time.time()

    text_encoder.train(); vision_encoder.train()
    txt_proj_layer.train(); proj_head.train(); scene_head.train()

    epoch_loss = 0.0

    for batch_idx, batch in enumerate(train_loader):
        pixel_values, input_ids, attention_mask, scene_labels = [b.to(device) for b in batch]
        loss = train_step(pixel_values, input_ids, attention_mask, scene_labels)
        epoch_loss += loss

        if (batch_idx + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] | Batch [{batch_idx+1}/{len(train_loader)}] | Loss: {loss:.4f}")

    avg_loss = epoch_loss / len(train_loader)
    elapsed = time.time() - start_time
    print(f"✅ Epoch {epoch+1} 완료! | 평균 Loss: {avg_loss:.4f} | 소요 시간: {elapsed:.0f}초\n")

print("💾 학습 완료! 가중치를 저장하고 구글 드라이브 동기화를 강제 실행합니다...")

# 확실하게 어댑터 가중치와 config 생성
text_encoder.save_pretrained(os.path.join(SAVE_DIR, "text_lora"))
vision_encoder.save_pretrained(os.path.join(SAVE_DIR, "vision_lora"))

heads_state = {
    'txt_proj_layer': txt_proj_layer.state_dict(),
    'proj_head': proj_head.state_dict(),
    'scene_head': scene_head.state_dict()
}
torch.save(heads_state, os.path.join(SAVE_DIR, "custom_heads.pt"))

# 💡 구글 드라이브에 파일이 즉시 쓰이도록 강제 동기화 (에러 원인 차단)
drive.flush_and_unmount()
drive.mount('/content/drive')

print(f"🎉 성공적으로 저장되었습니다! 이제 무조건 adapter_config.json이 존재합니다.")

🏃‍♂️ Stage 2 학습을 시작합니다! (총 5 Epochs)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
from google.colab import files

files.downloads(os.path.join(SAVE_DIR, "custom_heads.pt"))

In [ ]:
from google.colab import files

files.downloads(os.path.join(SAVE_DIR, "custom_heads.pt"))